# iNat ↔ Wikidata taxon matching — report

Results, negative-sampling story, and error analysis. See `docs/inat-wikidata-match-spec.md`.

This notebook is populated incrementally, one section per milestone (spec §7), as each lands —
not written retroactively at the end.

## Milestone 1 — Ingest

Read `~/.cache/wikidata-inat-checker/taxa.db` (built by the sibling
[wikidata-inat-checker](https://github.com/Livia-Rasp/wikidata-inat-checker) repo) read-only,
and build a cached normalised-name + FTS5 trigram lookup table from it. See `src/normalize.py`
and `src/candidates.py`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.candidates import (
    build_lookup_cache,
    lookup_by_normalized_name,
    DEFAULT_TAXA_DB_PATH,
    DEFAULT_CACHE_PATH,
)
from src.normalize import normalize_name

conn = build_lookup_cache()
print(f"taxa index:   {DEFAULT_TAXA_DB_PATH}")
print(f"lookup cache: {DEFAULT_CACHE_PATH}")

taxa index:   /home/livia/.cache/wikidata-inat-checker/taxa.db
lookup cache: /home/livia/repos/xgboost-inat-wikidata-match/data/lookup.sqlite


### Corpus size and rank distribution

Confirms the index matches what the ingest exploration found: 1.4M active taxa, `rank_level`
and `active` not persisted (dropped by the Node builder — flagged as a gap to close in
milestone 4, features).

In [2]:
total = conn.execute("SELECT COUNT(*) FROM taxa_normalized").fetchone()[0]
print(f"{total:,} taxa in the index\n")
print(f"{'rank':15s}count")
for rank, n in conn.execute(
    "SELECT rank, COUNT(*) c FROM taxa_normalized GROUP BY rank ORDER BY c DESC LIMIT 10"
):
    print(f"{rank:15s}{n:,}")

1,418,443 taxa in the index

rank           count
species        1,116,164
genus          143,042
subspecies     87,842
variety        18,859
family         10,719
subgenus       6,896
hybrid         6,518
tribe          6,070
subfamily      5,472
section        4,842


### The hemihomonym check (milestone 1's acceptance test)

`Prunella` names both a bird genus (accentors, Prunellidae) and a mint genus (self-heal,
Lamiaceae). Name-only matching can't tell them apart — only ancestry can. This is exactly the
case candidate strategy 1 (spec §2) must surface both candidates for, rather than stopping at
the first hit.

In [3]:
matches = lookup_by_normalized_name(conn, "prunella")
print(f"{len(matches)} match(es) for 'prunella':")
for row in matches:
    print(f"  {row}")

2 match(es) for 'prunella':
  {'taxon_id': '13982', 'name': 'Prunella', 'rank': 'genus', 'ancestry': '48460/1/2/355675/3/7251/71358'}
  {'taxon_id': '52765', 'name': 'Prunella', 'rank': 'genus', 'ancestry': '48460/47126/211194/47125/47124/48151/48623/520502/918917/919181'}


### Name normalisation examples (spec §1)

Authorship stripping, infraspecific connectors, hybrid markers, and the gender-stripped epithet
stem — the last of these is what makes `Acer rubrum` / `Acer ruber` collapse to the same stem
despite disagreeing on gender.

In [4]:
cases = [
    "Rosa canina L.",
    "Acer rubrum",
    "Acer ruber",
    "Rosa canina subsp. dumetorum",
    "Salix x sepulcralis",
    "\u00d7Fragaria",
    "Bellis perennis (L.) DC.",
]
for c in cases:
    n = normalize_name(c)
    print(f"{c!r:35} -> {n.normalized!r:35} stem={n.epithet_stem!r:12} hybrid={n.hybrid}")

'Rosa canina L.'                    -> 'rosa canina'                       stem='canin'      hybrid=False
'Acer rubrum'                       -> 'acer rubrum'                       stem='rubr'       hybrid=False
'Acer ruber'                        -> 'acer ruber'                        stem='rubr'       hybrid=False
'Rosa canina subsp. dumetorum'      -> 'rosa canina subsp dumetorum'       stem='canin'      hybrid=False
'Salix x sepulcralis'               -> 'salix sepulcralis'                 stem='sepulcral'  hybrid=True
'×Fragaria'                         -> 'fragaria'                          stem=None         hybrid=True
'Bellis perennis (L.) DC.'          -> 'bellis perennis'                   stem='perenn'     hybrid=False


### Observation: how much genuine name-collision exists, and one normalisation gap it surfaced

48,280 normalised names (3.4% of the index) are shared by 2+ taxa — the raw material for
hemihomonym-style ambiguity that candidate generation and the taxonomic-agreement features
(spec §4) need to resolve.

But the single most-duplicated name, `cortinarius` (501 rows), isn't really 501 genuine
homonyms — it's a normalisation gap. Provisional/unresolved iNat species like `Cortinarius sp.
'AZ19'` fail the authorship-boundary heuristic in `normalize.py` at `sp.` and get truncated back
to just the genus, `cortinarius`, colliding with the real genus-rank entry and each other. This
doesn't affect the milestone 1 acceptance check, but it inflates the homonym-group count above
and is worth a real decision before milestone 3 (candidate generation): either teach
`normalize.py` to recognise and drop provisional-name taxa specifically, or filter `sp.`/quoted
provisional epithets out of candidate generation entirely, since they have no stable Wikidata
counterpart to match against anyway.

In [5]:
homonym_groups = conn.execute(
    """
    SELECT COUNT(*) FROM (
        SELECT normalized_name FROM taxa_normalized GROUP BY normalized_name HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]
print(f"{homonym_groups:,} normalized names shared by 2+ taxa ({homonym_groups/total:.1%} of the index)")

dup = conn.execute(
    """
    SELECT normalized_name, COUNT(*) c FROM taxa_normalized
    GROUP BY normalized_name HAVING c > 1 ORDER BY c DESC LIMIT 5
    """
).fetchall()
print("most-duplicated normalized names:", dup)

sample = conn.execute(
    "SELECT taxon_id, name, rank FROM taxa_normalized WHERE normalized_name='cortinarius' AND rank='species' LIMIT 3"
).fetchall()
print("sample of what's actually behind 'cortinarius':", sample)

48,280 normalized names shared by 2+ taxa (3.4% of the index)
most-duplicated normalized names: [('cortinarius', 501), ('hygrocybe', 216), ('ramaria', 176), ('inocybe', 167), ('amanita', 161)]
sample of what's actually behind 'cortinarius': [('1666456', "Cortinarius sp. 'AZ19'", 'species'), ('1668233', "Cortinarius sp. 'PNW101'", 'species'), ('1662842', "Cortinarius sp. 'IN10'", 'species')]


## Milestone 2 — Wikidata pull

Batched SPARQL against `https://query.wikidata.org/sparql` for Wikidata taxa carrying P3151
(iNat taxon ID), `LIMIT`-capped to the spec's target size and cached to parquet. See
`src/wikidata.py`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.wikidata import build_pull_cache, DEFAULT_CACHE_PATH, DEFAULT_TARGET_SIZE

result = build_pull_cache()
print(f"target size:  {DEFAULT_TARGET_SIZE:,}")
print(f"cache:        {DEFAULT_CACHE_PATH}")
print(f"cache hit:    {result.cache_hit}")
print(f"rows pulled:  {len(result.taxa):,}")

target size:  60,000
cache:        /home/livia/repos/xgboost-inat-wikidata-match/data/wikidata_taxa.parquet
cache hit:    True
rows pulled:  58,064

### How much of Wikidata's P3151 population this actually covers

A live `COUNT` against the endpoint, for context on how small a slice the spec's 30k–60k target
size actually is.

In [2]:
from src.wikidata import _sparql_get_tsv

rows = _sparql_get_tsv(
    "SELECT (COUNT(?item) AS ?count) WHERE { ?item wdt:P31 wd:Q16521 . ?item wdt:P3151 ?inatId . }"
)
total_p3151 = int(rows[0]["count"])
print(f"{total_p3151:,} Wikidata items are taxa with P3151 set")
print(f"pulled {len(result.taxa):,} ({len(result.taxa)/total_p3151:.1%} of the total)")

856,040 Wikidata items are taxa with P3151 set
pulled 58,064 (6.8% of the total)

### Observation: most P3151 links carry no reference

Spec §3 flags that a chunk of P3151 statements were bot-added by exact-name matching, so
"ground truth" is partly the baseline this project is trying to beat, and its stated mitigation
is to prefer referenced statements. Checking reference presence turned up something starker than
"a chunk": **99.66% of pulled statements have no reference at all.** That's not itself proof any
particular link is wrong — most are presumably fine — but it means the reference-presence signal
alone can't carry much of the noise-mitigation weight `labels.py` will need later; the "hand-check
a random sample of 100" step spec §3 also calls for is doing most of the real work.

`has_commons_cat` and `has IUCN status` rates are shown for the same reason milestone 1's report
carries corpus stats: sanity-checking the pull before anything is built on top of it.

In [3]:
df = result.taxa
n = len(df)
print(f"{n:,} rows in the cache\n")
print(f"reference rate:   {df['p3151_has_reference'].sum():,}/{n:,} ({df['p3151_has_reference'].mean():.2%})")
print(f"commons cat rate: {df['has_commons_cat'].sum():,}/{n:,} ({df['has_commons_cat'].mean():.1%})")
print(f"has IUCN status:  {n - df['iucn_qid'].isna().sum():,}/{n:,} ({(1 - df['iucn_qid'].isna().mean()):.1%})")
print()
print("top rank_qid values (raw QID; label resolution deferred to milestone 4):")
print(df["rank_qid"].value_counts().head(6).to_string())

58,064 rows in the cache

reference rate:   198/58,064 (0.34%)
commons cat rate: 15,513/58,064 (26.7%)
has IUCN status:  18,395/58,064 (31.7%)

top rank_qid values (raw QID; label resolution deferred to milestone 4):
rank_qid
Q7432      46516
Q34740      6954
Q68947      2423
Q35409       856
Q767728      807
Q36602       172

### Cross-check against milestone 1's iNat index

Confirms the two caches actually join: each pulled `inat_id` should resolve to the same taxon
name in `data/lookup.sqlite`.

In [4]:
import sqlite3

conn_inat = sqlite3.connect("../data/lookup.sqlite")
sample = df[["qid", "inat_id", "name"]].head(3)
for _, row in sample.iterrows():
    inat_row = conn_inat.execute(
        "SELECT taxon_id, name, rank FROM taxa_normalized WHERE taxon_id=?", (row["inat_id"],)
    ).fetchone()
    print(f"{row['qid']} ({row['name']!r}) -> iNat taxon_id {row['inat_id']}: {inat_row}")

Q557493 ('Melanerpes uropygialis') -> iNat taxon_id 18161: ('18161', 'Melanerpes uropygialis', 'species')
Q569535 ('Melanerpes carolinus') -> iNat taxon_id 18205: ('18205', 'Melanerpes carolinus', 'species')
Q1263378 ('Odontophorus leucolaemus') -> iNat taxon_id 1371: ('1371', 'Odontophorus leucolaemus', 'species')